# GIADA45e — missing evaluations only
10 completed blocks reused,8 missing blocks evaluated from saved frozen checkpoints. Training, selection, thresholds and abstraction unchanged. Original full archive preserved; no native regeneration.

In [ ]:
import hashlib,json,os,shutil,subprocess,sys,zipfile,time,threading
from pathlib import Path
REVISION='PINNED_BY_CONSOLE_DISPATCH'
SOURCE_SHA='7e07984c6c8770fee223a83f4694922c2f8cb92529e3df899bc511cf8d5080be'
WORK=Path('/tmp/giada_task45e_eval_recovery');WORK.mkdir(exist_ok=False)
repo=WORK/'giada';out=WORK/'result';public=Path('/kaggle/working');log=public/'process.log'
code=None;started=time.time()
def setup(cmd):
 with log.open('a') as stream:subprocess.run(cmd,stdout=stream,stderr=subprocess.STDOUT,check=True)
try:
 assert len(REVISION)==40,'Use console dispatch to pin the exact commit'
 print('[GIADA41n recovery] pinned Torch/GPU setup',flush=True)
 setup([sys.executable,'-m','pip','install','--quiet','--upgrade','torch==2.11.0+cu130','--index-url','https://download.pytorch.org/whl/cu130'])
 for cmd in (['git','clone','--quiet','--depth','1','--branch','codex/surrogate-validity-audit','https://github.com/Zagred47/giada.git',str(repo)],['git','-C',str(repo),'fetch','--quiet','--depth','1','origin',REVISION],['git','-C',str(repo),'checkout','--quiet','--detach','FETCH_HEAD']):setup(cmd)
 sources=list(Path('/kaggle/input').rglob('source41n.zip.bin'));assert len(sources)==1
 with sources[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()==SOURCE_SHA
 native=list(Path('/kaggle/input').rglob('task45b_final.zip'));checkpoints=list(Path('/kaggle/input').rglob('checkpoint45c.zip.bin'));assert len(native)==len(checkpoints)==1
 with native[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()=='eee11b0924fe6bde6b8851afe8e6ec166f3a94711420db81dc66d6f88becc6da'
 with checkpoints[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()=='555133b7fc71719ecf459e2b04ffed8fd22a50af198bdb409735e0d8ca996afe'
 saved=list(Path('/kaggle/input').rglob('task45e_final.zip'));assert len(saved)==1
 with saved[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()=='16cf035671bdc4fd877a381db3bfdbf3a7a448e6d075bfb811847f41730180a8'
 env=dict(os.environ,CUBLAS_WORKSPACE_CONFIG=':4096:8',MPLBACKEND='Agg')
 with log.open('a') as f:
  proc=subprocess.Popen([sys.executable,'-u',str(repo/'scripts/run_task45e_evaluation_recovery.py'),str(sources[0]),str(native[0]),str(saved[0]),str(checkpoints[0]),str(out)],cwd=repo,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  timer=threading.Timer(10800,proc.terminate);timer.daemon=True;timer.start()
  for line in proc.stdout:
   f.write(line);f.flush()
   if line.startswith('[GIADA') or 'Error' in line or 'Traceback' in line:print(line.rstrip(),flush=True)
  code=proc.wait();timer.cancel()
finally:
 out.mkdir(exist_ok=True)
 if log.exists():shutil.copy2(log,out/'process.log')
 (out/'runtime_receipt.json').write_text(json.dumps(dict(revision=REVISION,source_sha256=SOURCE_SHA,worker_exit_code=code,elapsed_seconds=time.time()-started,training_performed=False,selection_recomputed=False,evaluation_only=True,thresholds_changed=False)))
 archive=Path(shutil.make_archive(str(public/'task45e_eval_final'),'zip',out))
 with archive.open('rb') as f:digest=hashlib.file_digest(f,'sha256').hexdigest()
 (public/'kaggle_receipt.json').write_text(json.dumps(dict(valid=code==0,revision=REVISION,source_sha256=SOURCE_SHA,archive=archive.name,sha256=digest,bytes=archive.stat().st_size,worker_exit_code=code,training_performed=False,selection_recomputed=False,evaluation_only=True,thresholds_changed=False)))
 print('[GIADA41n recovery] final archive persisted '+digest,flush=True)
assert code==0,'Evaluation recovery failed technically; preserve output and logs'
